# Module 3.1: SSD Hardware Interfaces, Power Delivery & Partitioning

## Homelab NAS Development - Step 3: Storage Expansion

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Assess hardware storage configuration: WD Blue SN5000 NVMe SSD + UGREEN M.2 Enclosure over USB 3.0.
2. Understand UASP (USB Attached SCSI Protocol) vs. BOT (Bulk-Only Transport) performance modes.
3. Detect block devices and evaluate storage health using Linux disk utilities.
4. Partition the external Solid State Drive and format it with an optimized ext4 filesystem.

## 3.1.1 Storage Architecture for Homelab NAS

Immich photo and video ingestion requires high I/O throughput for thumbnail generation, database indexing, and raw file uploads. Running storage directly off a MicroSD card causes bottlenecks and premature card failure.

### Your Hardware Configuration: WD Blue SN5000 + UGREEN Enclosure
- **SSD**: Western Digital WD Blue SN5000 PCIe Gen 4 x4 M.2 2280 NVMe SSD.
- **Enclosure**: UGREEN 10Gbps M.2 NVMe SSD Aluminum Enclosure with thermal pad and heatsink ridges.
- **Bus Interface**: Connect strictly to one of the two **blue USB 3.0 ports** on the Raspberry Pi 4. The black USB 2.0 ports are bandwidth-limited to ~40 MB/s.

```
[ Raspberry Pi 4 (Blue USB 3.0 Port) ]
                 |
         (USB 3.0 / 5 Gbps Bus)
                 |
     [ UGREEN Aluminum Enclosure ]
                 |
    [ WD Blue SN5000 NVMe SSD ] ---> Formatted as ext4 with journaling & TRIM support
```

### UASP (USB Attached SCSI Protocol) vs. BOT
- **BOT (Bulk-Only Transport)**: Legacy USB storage protocol that processes disk commands sequentially (one command at a time). Maximum throughput is bottlenecked around ~200 MB/s with high CPU overhead.
- **UASP (USB Attached SCSI Protocol)**: Modern protocol that supports command queuing (up to 32 commands concurrently) and out-of-order execution. Delivers ~400+ MB/s on USB 3.0.
- Verify UASP operation using:
  ```bash
  lsusb -t
  ```
  Look for `Driver=uas` on the USB 3.0 bus. (If it displays `Driver=usb-storage`, the kernel fell back to BOT mode).

### Filesystem Rationale: ext4 vs. Btrfs / ZFS on Pi 4
`ext4` is the recommended filesystem for single-drive Raspberry Pi setups. Advanced filesystems like Btrfs or ZFS introduce significant memory and CPU overhead for copy-on-write checksumming and snapshots, which can exhaust RAM on a 4GB board. `ext4` provides rock-solid journaling, maximum speed, and minimal memory overhead.

## 📖 Official Hardware & Setup Reference Figures

<br>

<img src="../../assets/hardware/wd_blue_sn5000.jpg" width="75%">

*Reference Figure 1: Western Digital WD Blue SN5000 M.2 2280 NVMe SSD (Your Primary Drive)*

<br>

<br>

<img src="../../assets/hardware/ugreen_nvme_enclosure.jpg" width="75%">

*Reference Figure 2: UGREEN 10Gbps M.2 NVMe SSD Aluminum Enclosure (Your Enclosure)*

<br>

## 3.1.2 Step-by-Step Laboratory Guide

### Task 3.1.1: Assemble Hardware & Identify Block Device
1. Install your WD Blue SN5000 into the UGREEN enclosure with the included thermal silicone pad.
2. Plug the enclosure cable into one of the **blue USB 3.0 ports** on your Raspberry Pi 4.
3. Check kernel device detection:
```bash
lsblk -o NAME,FSTYPE,SIZE,MOUNTPOINT,MODEL,TRAN
sudo dmesg | grep -i usb | tail -n 20
```
Locate your external disk (usually `/dev/sda`). **Ensure you do not select your OS drive (`/dev/mmcblk0`)!**

### Task 3.1.2: Partition and Format the Disk
Use `parted` to create a fresh GPT partition table:
```bash
# WARNING: Ensure /dev/sda is your external SSD!
sudo parted /dev/sda --script mklabel gpt
sudo parted /dev/sda --script mkpart primary ext4 0% 100%
```
Format the new partition with `ext4` and assign an identifier label:
```bash
sudo mkfs.ext4 -L homelab_data -m 1 /dev/sda1
```
*Note: The `-m 1` flag reduces reserved root filesystem blocks from 5% to 1%, liberating gigabytes of storage for media files.*

## 📸 Verification & Screenshot Submissions

*To confirm milestone completion, capture proof of your progress and insert your screenshots in the designated placeholders below.*

### 🖼️ Milestone Submission 1: Block Device List (`lsblk`) with Formatted SSD

**Verification Objective:** Capture a screenshot of your terminal executing `lsblk -f` showing the SSD partitioned and formatted with the `homelab_data` label.

---

![Block Device List (`lsblk`) with Formatted SSD](assets/screenshots/03_1_lsblk_output.png)

*Figure 1: Verification evidence for Block Device List (`lsblk`) with Formatted SSD. Place your image at `assets/screenshots/03_1_lsblk_output.png`.*

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] All laboratory tasks executed and validated without unhandled errors.
- [ ] Key configurations and credentials confirmed.
- [ ] Verification screenshot captured and referenced.

Proceed to the next module in sequence once this checkpoint is satisfied.